# CivicQ — Talk to Government Data | GenAI Intern Screening

## A Natural-Language Analyst for US Air Quality Data

### Design Note


**1. Stack & Decisions**

| Component | Choice | Rationale |
|---|---|---|
| **LLM** | DeepSeek (deepseek-chat) via OpenAI-compatible API | ~$0.28 total. Strong structured JSON output. No need for LangChain/LlamaIndex — orchestration simple enough to hand-roll. |
| **Data layer** | pandas | Single CSV ~30 MB; pandas sufficient and universally understood. DuckDB better at scale. |
| **Framework** | Hand-rolled | Pipeline: LLM → JSON → pandas → answer. No framework abstraction needed at this size. |
| **UI** | Gradio | Single-cell launch in Colab with share=True. Fastest working UI connected to backend. |
| **Query approach** | Constrained structured JSON DSL → pandas | LLM emits JSON describing the query operation, filters, grouping. Translated to pandas — no exec(). Trade-off: DSL limits expressiveness (no joins, multi-step transforms), but eliminates code injection risk. Covers all realistic questions for this dataset's scope. |


**2. Correctness & Trust** — Three layers prevent hallucination:

1. **The LLM never computes.** It only translates NL into JSON DSL. Real computation runs on pandas against actual data. LLM can't fabricate numbers.
2. **The DSL is validated** — unknown operations, invalid columns, or missing required fields trigger clear errors.
3. **Provenance is surfaced.** Every answer shows the exact JSON DSL that ran. A sceptical user can verify logic. For government: add dataset row-count hash, signed query log, and reproducibility button.


**3. Government Deployment** — Prototype to production:

- **Data residency:** Dataset lives in government-controlled DB (PostgreSQL/SQLite), not downloaded at runtime. LLM endpoint on-prem or FedRAMP-authorized.
- **Audit trail:** Every question, DSL, result hash, timestamp, and user identity logged to append-only store. No deletion.
- **Access control:** Row-level security. User in Gujarat only queries Gujarat data unless authorized otherwise. DSL translator injects user-scoped filters automatically.
- **Generated code risk:** exec() avoided entirely via DSL. If arbitrary queries needed: sandbox in read-only DuckDB with filesystem isolation, timeout, row limits.


**4. Scaling** — First two things that break going from 1 CSV to hundreds of tables / millions of rows:

1. **Schema discovery.** DSL schema currently hand-coded. With hundreds of tables, LLM needs schema-aware step — vector search over column metadata or dynamic schema injection into prompt.
2. **Query performance.** pandas loads entire dataset in memory. Switch to DuckDB — SQL pushdown, in-process, handles larger-than-RAM via spilling. DSL translator emits SQL instead of pandas. Pattern stays the same, engine changes.


**5. Validation with Non-Technical Stakeholders**

- **Guided walkthrough.** Sit with 3-5 stakeholders (policy analyst, data clerk, decision-maker). 5 predefined questions + their own. Watch: do they trust the answer? Understand provenance? Where do they hesitate?
- **Failure mode observation.** Questions almost in-scope but slightly off (e.g., "AQI for NYC" when data has "New York County"). How does system handle ambiguity?
- **Synthetic edge cases.** Misspellings, multiple intents ("AQI in California and population in Texas"), questions requiring data not in dataset.


**6. Honest Limitations**

- **Single dataset, single source.** No joins across domains. "Does AQI correlate with asthma rates?" cannot be answered.
- **No disambiguation.** "New York" could be state or city. LLM guesses — no interactive clarification.
- **Static time range.** Only 2022-2023. Questions about 2024 silently empty unless data added.
- **Chart quality.** Matplotlib. Production would use Plotly for interactivity.
- **No confidence scoring.** Low-data questions (county with 10 reporting days) get same presentation as high-confidence answers.


## Setup — Install Dependencies


In [ ]:
!pip install openai pandas gradio matplotlib seaborn requests -q
print("Dependencies installed.")


## API Key Setup

Set your DeepSeek API key. Choose ONE:
- **Colab:** Use Secrets panel (left sidebar, key icon) → add `DEEPSEEK_API_KEY`
- **Local:** getpass will prompt

Get free key at https://platform.deepseek.com


In [ ]:
import os

DEEPSEEK_API_KEY = None
try:
    from google.colab import userdata
    DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
except (ImportError, ValueError):
    pass

if not DEEPSEEK_API_KEY:
    from getpass import getpass
    DEEPSEEK_API_KEY = getpass("Enter DeepSeek API key: ")

os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY
print("API key:", DEEPSEEK_API_KEY[:8], "...")


## Download & Load the Dataset

Using US EPA Air Quality Index (AQI) data — a US government open dataset.

**Source:** https://www.epa.gov/outdoor-air-quality-data
**License:** Public Domain (US Government Work)


In [ ]:
import requests, zipfile, io
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

for year in [2022, 2023]:
    url = f"https://aqs.epa.gov/aqsweb/airdata/daily_aqi_by_county_{year}.zip"
    print(f"Downloading {url}...")
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(resp.content))
    z.extractall(DATA_DIR)
    print(f"  Extracted {len(z.namelist())} files.")

print("Download complete.")


In [ ]:
import pandas as pd
import numpy as np

df_list = []
for year in [2022, 2023]:
    f = DATA_DIR / f"daily_aqi_by_county_{year}.csv"
    df_year = pd.read_csv(f, low_memory=False)
    df_year["year"] = year
    df_list.append(df_year)

df = pd.concat(df_list, ignore_index=True)
print(f"Loaded {len(df):,} rows, {df["state_name"].nunique()} states, {df["county_name"].nunique()} counties")


In [ ]:
# Standardize column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

# Parse dates
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Drop rows with missing critical fields
before = len(df)
df = df.dropna(subset=["date", "aqi", "state_name"])
print(f"Dropped {before - len(df):,} rows")

# Derived columns
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.month_name()
df["weekday"] = df["date"].dt.day_name()

def get_season(m):
    if m in (12, 1, 2): return "Winter"
    if m in (3, 4, 5): return "Spring"
    if m in (6, 7, 8): return "Summer"
    return "Fall"

df["season"] = df["month"].apply(get_season)

# Clean string columns
for col in ["state_name", "county_name", "category", "defining_parameter"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()
        df[col] = df[col].replace({"Nan": None, "N/A": None, "": None})

# AQI category rank for sorting
df["aqi_category_rank"] = df["category"].map({
    "Good": 1,
    "Moderate": 2,
    "Unhealthy For Sensitive Groups": 3,
    "Unhealthy": 4,
    "Very Unhealthy": 5,
    "Hazardous": 6
})

# Normalize pollutants
df["defining_parameter"] = df["defining_parameter"].str.upper()

print(f"\nFinal: {len(df):,} rows, {df["year"].min()}-{df["year"].max()}")
print(f"Columns: {list(df.columns)}")
print(f"Categories: {df["category"].value_counts().to_dict()}")


## DeepSeek LLM Client


In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com"
)

# Quick test
resp = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "Say hello."}],
    temperature=0,
    max_tokens=20
)
print(resp.choices[0].message.content)


## JSON Query DSL — The Structured Query Language

LLM emits constrained JSON (never raw Python). Covers all common analytical operations, translated to pandas safely — no exec().


In [ ]:
import json

SYSTEM_PROMPT = """You translate natural language questions into structured JSON queries for an air quality dataset.

The dataset contains daily AQI readings by US county and state.

Available columns:
- state_name: US state name (e.g., California, Texas)
- county_name: County name within the state
- date: Date of reading (YYYY-MM-DD)
- aqi: Air Quality Index (0-500, lower is better)
- category: AQI category (Good, Moderate, Unhealthy for Sensitive Groups, Unhealthy, Very Unhealthy, Hazardous)
- defining_parameter: Main pollutant (OZONE, PM2.5, PM10, CO, NO2, SO2)
- year: Calendar year (2022 or 2023)
- month: Month number (1-12)
- month_name: Month name
- season: Season (Winter, Spring, Summer, Fall)
- weekday: Day of week name

Rules:
1. Return ONLY valid JSON — no markdown, no explanation.
2. If UNANSWERABLE from this dataset, return {"error": "explanation"}.
3. For comparisons between entities, use operation "compare".
4. For ranking (best/worst/top), use operation "top_k" with order_by.
5. For trends over time, use operation "trend" with time_granularity.
6. For simple aggregations, use operation "aggregate".
7. For percentage calculations, use operation "percentage".
8. Use filters to narrow by state, county, year, category.
9. When a specific pollutant is mentioned, use that as the metric.

Schema:
{
  "operation": "aggregate | trend | compare | top_k | filter | percentage",
  "metric": "aqi | pm2.5 | pm10 | ozone | co | no2 | so2 | null",
  "group_by": "list of column names or null",
  "aggregation": "mean | max | min | sum | count | null",
  "filters": [
    {
      "column": "col",
      "op": "eq|neq|in|gt|lt|gte|lte",
      "value": "scalar|list"
    }
  ],
  "order_by": {
    "column": "...",
    "direction": "asc|desc"
  },
  "limit": "int or null",
  "time_granularity": "month|year|season|null"
}
"""


## Query Translator — JSON DSL → pandas

Core safety layer. Converts LLM's structured JSON into pandas. No exec().


In [ ]:
def translate_query(dsl, dataframe):
    result = dataframe.copy()

    # Apply filters
    for f in dsl.get("filters", []):
        col = f.get("column")
        op = f.get("op")
        val = f.get("value")
        if col not in result.columns:
            continue
        if op == "eq":
            result = result[result[col] == val]
        elif op == "neq":
            result = result[result[col] != val]
        elif op == "in":
            result = result[result[col].isin(val)]
        elif op == "gt":
            result = result[result[col] > val]
        elif op == "lt":
            result = result[result[col] < val]
        elif op == "gte":
            result = result[result[col] >= val]
        elif op == "lte":
            result = result[result[col] <= val]

    if result.empty:
        return result

    operation = dsl.get("operation", "aggregate")
    agg = dsl.get("aggregation", "mean")
    group_by = dsl.get("group_by") or []
    metric_raw = dsl.get("metric") or "aqi"
    metric = metric_raw.lower().replace(".", "_").replace(" ", "_")
    if metric not in result.columns:
        metric = "aqi"

    if operation == "filter":
        return result

    # Trend over time
    if operation == "trend":
        tg = dsl.get("time_granularity", "month")
        pmap = {
            "month": result["date"].dt.to_period("M").astype(str),
            "year": result["year"].astype(str),
            "season": result.apply(lambda r: f"{r["year"]}-{r["season"]}", axis=1),
        }
        if tg in pmap:
            result = result.copy()
            result["period"] = pmap[tg]
        gcols = ["period"]
        if group_by:
            gcols = group_by + gcols
        result = result.groupby(gcols)[metric].agg(agg).reset_index()
        result = result.sort_values("period")
        return result

    # Percentage
    if operation == "percentage":
        f_n = dsl.get("numerator_filters", [])
        f_d = dsl.get("denominator_filters", [])
        ndf = dataframe.copy()
        ddf = dataframe.copy()
        for f in f_n:
            c, o, v = f["column"], f["op"], f["value"]
            if o == "eq": ndf = ndf[ndf[c] == v]
            elif o == "in": ndf = ndf[ndf[c].isin(v)]
        for f in f_d:
            c, o, v = f["column"], f["op"], f["value"]
            if o == "eq": ddf = ddf[ddf[c] == v]
            elif o == "in": ddf = ddf[ddf[c].isin(v)]
        if len(ddf) == 0:
            return pd.DataFrame({"percentage": [0]})
        return pd.DataFrame({"percentage": [round(len(ndf)/len(ddf)*100, 1)]})

    # Aggregate / Compare / Top_K
    if group_by:
        gcols = [g for g in group_by if g in result.columns]
        if gcols:
            result = result.groupby(gcols)[metric].agg(agg).reset_index()
        else:
            result = result[[metric]].agg(agg).to_frame().T
    else:
        result = result[[metric]].agg(agg).to_frame().T

    if dsl.get("order_by"):
        ob = dsl["order_by"]
        c = ob.get("column", metric)
        result = result.sort_values(c, ascending=ob.get("direction") == "asc")

    if dsl.get("limit"):
        result = result.head(dsl["limit"])

    return result

# Test
test_dsl = {
    "operation": "top_k", "metric": "aqi",
    "group_by": ["state_name"], "aggregation": "mean",
    "order_by": {"column": "aqi", "direction": "desc"}, "limit": 5
}
print("Top 5 states by avg AQI:")
print(translate_query(test_dsl, df))


## Chart Generator


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-darkgrid")

def generate_chart(result_df, dsl):
    if result_df is None or result_df.empty:
        return None

    op = dsl.get("operation", "aggregate")
    agg = dsl.get("aggregation", "mean")
    metric = (dsl.get("metric") or "aqi").lower().replace(".", "_")
    if metric not in result_df.columns:
        nums = result_df.select_dtypes(include=[np.number]).columns
        metric = nums[0] if len(nums) > 0 else "aqi"

    fig, ax = plt.subplots(figsize=(10, 5))

    if op == "trend":
        x = "period" if "period" in result_df.columns else result_df.columns[0]
        extra = [c for c in result_df.columns if c not in (x, metric)]
        hue = extra[0] if extra else None
        if hue and result_df[hue].nunique() in range(2, 16):
            for g in sorted(result_df[hue].unique()):
                s = result_df[result_df[hue] == g]
                ax.plot(s[x].astype(str), s[metric], marker="o", label=g, linewidth=2, ms=4)
            ax.legend(frameon=True, facecolor="white", edgecolor="none", fontsize=9)
        else:
            ax.plot(result_df[x].astype(str), result_df[metric],
                    marker="o", color="#2563eb", linewidth=2.5, ms=5)
        ax.tick_params(axis="x", rotation=45)

    elif op in ("compare", "top_k", "aggregate"):
        cats = [c for c in result_df.columns if c != metric and result_df[c].dtype == "object"]
        cat = cats[0] if cats else None
        if cat and metric in result_df.columns:
            n = len(result_df)
            colors = plt.cm.Blues(np.linspace(0.35, 0.85, n))[::-1]
            bars = ax.bar(result_df[cat].astype(str), result_df[metric],
                          color=colors, edgecolor="white", linewidth=0.5, width=0.65)
            for b, v in zip(bars, result_df[metric]):
                ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{v:.1f}",
                        ha="center", va="bottom", fontsize=9, fontweight="bold")
            ax.tick_params(axis="x", rotation=45)

    elif op == "percentage":
        v = result_df["percentage"].iloc[0] if "percentage" in result_df.columns else 0
        ax.barh(["Result"], [v], color="#2563eb", height=0.4)
        ax.set_xlim(0, 100)
        ax.text(v+1, 0, f"{v:.1f}%", va="center", fontsize=14, fontweight="bold")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#e5e7eb")
    ax.spines["bottom"].set_color("#e5e7eb")
    ax.tick_params(colors="#6b7280")
    ml = metric.replace("_", " ").title()
    ax.set_title(f"{agg.title()} {ml}", fontsize=14, fontweight="bold", pad=12)
    ax.set_ylabel(f"{ml} ({agg})", fontsize=10)
    plt.tight_layout()
    return fig


## Orchestrator — Tie Everything Together


In [ ]:
def generate_answer_from_llm(result_df, question, dsl):
    if result_df.empty:
        return "No results returned."
    p = result_df.head(15).to_string()
    d = json.dumps(dsl, indent=2)
    prompt = ("Given query result from EPA AQI data, answer concisely.\n\n"
              + f"Question: {question}\n\n"
              + f"Query: {d}\n\n"
              + f"Results:\n{p}\n\n"
              + "Short accurate answer with key numbers.\nAnswer:")
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1, max_tokens=250
    )
    return resp.choices[0].message.content.strip()


def answer_question(question):
    try:
        resp = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": "Question: " + question + "\n\nReturn ONLY JSON."}
            ],
            response_format={"type": "json_object"},
            temperature=0.05,
            max_tokens=600
        )
        raw = resp.choices[0].message.content.strip()
        dsl = json.loads(raw)

        if "error" in dsl:
            return {"success": True, "answer": dsl["error"], "chart": None,
                    "provenance": "Out of scope: " + dsl["error"], "dsl": dsl}

        valid_ops = ("aggregate","trend","compare","top_k","filter","percentage")
        if dsl.get("operation") not in valid_ops:
            return {"success": False, "answer": "Invalid operation: "+str(dsl.get("operation")),
                    "chart": None, "provenance": "DSL validation failed.", "dsl": dsl}

        result_df = translate_query(dsl, df)
        if result_df.empty:
            return {"success": True, "answer": "No results. Try different year/state/pollutant.",
                    "chart": None, "provenance": "Empty result.\nDSL: "+json.dumps(dsl,indent=2), "dsl": dsl}

        chart = generate_chart(result_df, dsl)
        answer = generate_answer_from_llm(result_df, question, dsl)
        prov = ("## Query Executed\n\n```json\n"+json.dumps(dsl,indent=2)
                +"\n```\n\n**Rows:** "+str(len(result_df))
                +"\n\n**Data:**\n```\n"+result_df.head(10).to_string()+"\n```")

        return {"success": True, "answer": answer, "chart": chart, "provenance": prov, "dsl": dsl}

    except json.JSONDecodeError:
        return {"success": False, "answer": "LLM response not valid JSON.",
                "chart": None, "provenance": "JSON parse error.", "dsl": {}}
    except Exception as e:
        return {"success": False, "answer": f"Error: {type(e).__name__}: {str(e)}",
                "chart": None, "provenance": f"Exception: {type(e).__name__}: {str(e)}", "dsl": {}}


## Gradio UI

Launch the interactive web interface. Gives a public share link (valid ~72 hours).


In [ ]:
import gradio as gr

def handle(question):
    if not question or not question.strip():
        return "Please enter a question.", None, ""
    r = answer_question(question.strip())
    return r["answer"], r.get("chart"), r.get("provenance", "")

with gr.Blocks(title="CivicQ AQI Analyst",
                theme=gr.themes.Soft(primary_hue="blue", neutral_hue="stone")) as demo:
    gr.HTML("<div style=\"text-align:center;padding:1rem 0 0.5rem;\">            <h1 style=\"font-size:1.8rem;font-weight:700;letter-spacing:-0.02em;\">CivicQ AQI Analyst</h1>            <p style=\"color:#6b7280;font-size:0.9rem;\">Ask about US air quality in plain English.</p></div>")

    with gr.Row():
        q = gr.Textbox(label="Your Question", scale=4,
                       placeholder="e.g., Which state had highest AQI in 2023?")
        btn = gr.Button("Ask", variant="primary", scale=1, min_width=100)

    a = gr.Markdown(label="Answer")
    c = gr.Plot(label="Chart")
    with gr.Accordion("Provenance — Executed Query", open=False):
        p = gr.Markdown()

    examples = [
        "Which state had the worst average AQI in 2023?",
        "How did California AQI trend month-over-month in 2023?",
        "Compare average AQI between New York and California in 2023.",
        "Top 5 states with the best air quality in 2023.",
        "What percentage of days were Unhealthy in Texas in 2023?",
        "What was the GDP of India in 2023?",
    ]

    gr.Markdown("### Try examples")
    for ex in examples:
        b = gr.Button(ex, size="sm", variant="secondary")
        b.click(fn=lambda e=ex: e, outputs=q).then(fn=handle, inputs=q, outputs=[a, c, p])

    btn.click(fn=handle, inputs=q, outputs=[a, c, p])
    q.submit(fn=handle, inputs=q, outputs=[a, c, p])

print("\nLaunching Gradio...")
demo.launch(share=True, debug=False)
print("\nInterface running. Click URL above.")


## Evaluation Suite

5-8 questions with expected vs actual results. Includes out-of-scope trap and hand-verified number.


In [ ]:
eval_questions = [
    {"question": "Which state had the worst average AQI in 2023?",
     "type": "top_k", "expected_behavior": "Returns state with highest mean AQI", "hand_verified": True},
    {"question": "How did California AQI trend month-over-month in 2023?",
     "type": "trend", "expected_behavior": "Line chart, 12 monthly points", "hand_verified": False},
    {"question": "Compare average AQI between New York and California in 2023.",
     "type": "compare", "expected_behavior": "Bar chart, two bars", "hand_verified": False},
    {"question": "Top 5 states with the best air quality in 2023.",
     "type": "top_k", "expected_behavior": "5 states, lowest AQI ascending", "hand_verified": False},
    {"question": "What percentage of days in Texas were Unhealthy in 2023?",
     "type": "percentage", "expected_behavior": "Single percentage number", "hand_verified": False},
    {"question": "Which pollutant was most common on Unhealthy+ days in 2023?",
     "type": "aggregate", "expected_behavior": "Pollutant name + count", "hand_verified": False},
    {"question": "What was the GDP of India in 2023?",
     "type": "out_of_scope", "expected_behavior": "REFUSED — must not fabricate", "hand_verified": True},
    {"question": "How did AQI in Los Angeles County CA change from 2022 to 2023?",
     "type": "trend", "expected_behavior": "Two-year comparison", "hand_verified": False},
]

print(f"Loaded {len(eval_questions)} questions:")
print(f"  {sum(1 for q in eval_questions if q["hand_verified"])} hand-verified")
print(f"  {sum(1 for q in eval_questions if q["type"] == "out_of_scope")} out-of-scope trap")


In [ ]:
import time
results = []
for i, eq in enumerate(eval_questions):
    print(f"\n[{i+1}/{len(eval_questions)}] {eq["question"][:60]}...")
    try:
        r = answer_question(eq["question"])
        results.append({"question": eq["question"], "type": eq["type"],
                        "expected": eq["expected_behavior"],
                        "actual": r["answer"][:200],
                        "chart": r["chart"] is not None,
                        "pass": r["success"], "hv": eq["hand_verified"]})
        status = "PASS" if r["success"] else "FAIL"
        chart_ok = "Y" if r["chart"] else "N"
        print(f"  -> {status} | Chart: {chart_ok}")
    except Exception as e:
        results.append({"question": eq["question"], "type": eq["type"],
                        "expected": eq["expected_behavior"],
                        "actual": f"ERROR: {str(e)}",
                        "chart": False, "pass": False, "hv": eq["hand_verified"]})
        print(f"  -> ERROR: {str(e)[:50]}")
    time.sleep(0.5)

print(f"\nEvaluation: {sum(1 for r in results if r["pass"])}/{len(results)} passed")


In [ ]:
from IPython.display import display
print("\n=== Results ===")
for r in results:
    tag = " [HAND-VERIFIED]" if r["hv"] else ""
    print(f"\n{\"PASS\" if r[\"pass\"] else \"FAIL\"}{tag}")
    print(f"  Q: {r["question"]}")
    print(f"  Expected: {r["expected"]}")
    print(f"  Actual: {r["actual"][:200]}")

import pandas as pd
display(pd.DataFrame(results)[["question", "type", "chart", "pass"]])


## Decisions Log

### Trade-offs

| Decision | Alternative | Chosen & Why |
|---|---|---|
| LLM | Groq, Gemini, Claude, local | DeepSeek — cheapest with strong JSON output |
| Dataset | CPCB India (data.gov.in) | EPA — direct CSV, no API needed, clean schema |
| Query | Raw exec(), LangChain SQL | JSON DSL → pandas — no code injection risk |
| UI | Streamlit, FastAPI+React | Gradio — one cell, built-in share=True |
| Answer gen | Templates vs LLM | LLM from results at temp=0.1 for flexibility |

### Where I got stuck

1. **Column name normalization** — EPA CSV columns have spaces/hyphens. Must standardize before DSL translator works reliably.
2. **Percentage operation** — Needed special DSL path (two filter operations + ratio). Could simplify with post-processing on count aggregations.
3. **Chart axis rotation** — Long state names overlap. Fixed with rotation=45. Production: horizontal bars for long labels.
